In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pickle
import numpy as np
from bm3d import bm3d
from tqdm import tqdm
from torchvision import transforms
import os
import pandas as pd

from HARUnet_model_v2_1 import HARU_net

from ResUNet_model import ResUNet


#from torch.cuda.amp import GradScaler, autocast
#from torchmetrics.image.ssim import StructuralSimilarityIndexMeasure as ssim
from utils import psnr, batch_psnr

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
# Load data
def load_data(pickle_file):
    with open(pickle_file, 'rb') as f:
        patches = pickle.load(f)
    return patches  # Expecting a NumPy array (N, 1, H, W)

class CBCTDataset(Dataset):
    def __init__(self, noisy_patches, target_patches):
        self.noisy = noisy_patches  # Keep as is
        self.target = target_patches  # Keep as is

    def __len__(self):
        return len(self.noisy)

    def __getitem__(self, idx):
        noisy_tensor = torch.tensor(self.noisy[idx], dtype=torch.float32)
        target_tensor = torch.tensor(self.target[idx], dtype=torch.float32)

        return noisy_tensor, target_tensor
    

In [3]:
pickled_train_inputs = r"D:\CBCT_Denoising\Pickled CBCT Data\MoreNoisy\Train_noisyCBCT_patches.pkl"
pickled_train_targets = r"D:\CBCT_Denoising\Pickled CBCT Data\MoreNoisy\Train_origCBCT_patches.pkl"
train_inputs = load_data(pickled_train_inputs)  # Shape: (N, 1, H, W)
train_targets = load_data(pickled_train_targets)  # Shape: (N, 1, H, W)

train_dataset = CBCTDataset(train_inputs, train_targets)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0, pin_memory=True)

pickled_val_inputs = r"D:\CBCT_Denoising\Pickled CBCT Data\MoreNoisy\Val_noisyCBCT_patches.pkl"
pickled_val_targets = r"D:\CBCT_Denoising\Pickled CBCT Data\MoreNoisy\Val_origCBCT_patches.pkl"
val_inputs = load_data(pickled_val_inputs)  # Shape: (N, 1, H, W)
val_targets = load_data(pickled_val_targets)  # Shape: (N, 1, H, W)

val_dataset = CBCTDataset(val_inputs, val_targets)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=True, num_workers=0, pin_memory=True)

In [4]:
class DistillationLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.hard = nn.MSELoss()
        self.soft = nn.MSELoss()

    def forward(self, student_out, teacher, targets, alpha):
        
        hard_loss = self.hard(student_out, targets)
        soft_loss = self.soft(student_out, teacher)

        return alpha * soft_loss + (1 - alpha) * hard_loss
    
criterion_distillation = DistillationLoss()
criterion_inference = nn.MSELoss()


In [5]:
model_dir = r"C:\Users\au711969\OneDrive - Aarhus universitet\Dentistry_Stuff\My Research projects\CBCT Denoising Project\BM3D_on_CBCT\Codes"

HARUnet_modelname = r"HARUnetv2_11_trainedon_noisytorawCBCTs_CadavarData_at_42epochs_.pth"
teacher_model = torch.load(os.path.join(model_dir,HARUnet_modelname)).to(device)

teacher_model = teacher_model.to(device)

DistHARU2ResUnet_modelname = r"DistilledHARUnet_ResUNet_trainedon_CBCT_CadavarData_at_50epochs_.pth"
student_model = torch.load(os.path.join(model_dir,DistHARU2ResUnet_modelname))
student_model = student_model.module
#
student_model = student_model.to(device)

optimizer = optim.Adam(student_model.parameters(), lr=1e-6)

C:\Users\au711969\AppData\Local\Temp\ipykernel_24492\1692082829.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  teacher_model = torch.load(os.path.join(model_dir,HARUnet

In [6]:
import torch.nn.utils.prune as prune


In [7]:
    
train_loss_history = []
train_inpsnr_history = []
train_outpsnr_history = []

val_loss_history = []
val_inpsnr_history = []
val_outpsnr_history = []

teacher_model.eval()
# -----------------------------
# 3) Iterative pruning loop
# -----------------------------
num_iterations = 5
prune_amount = 0.05  # prune 5% each iteration
epochs = num_iterations

for itr in range(num_iterations):

    for name, module in student_model.named_modules():
            if isinstance(module, nn.Conv2d):
                prune.ln_structured(
                    module,
                    name="weight",
                    amount=prune_amount,
                    n=2,       # L2 norm
                    dim=0      # prune output channels (filters)
                )
                print(f"Pruned {((itr+1)*prune_amount)*100:.1f}% filters from {name}")


    epochs = 10
    for epoch in range(epochs):

    ###################### Pruning Code #########################

        train_loss = 0.0
        train_inpsnr = 0
        train_outpsnr = 0

        train_loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
        alpha = 1

        ###################### Training Code #########################
        student_model.train()
        for train_inputs, train_targets in train_loop:

            train_inputs, train_targets = train_inputs.unsqueeze(1).to(device), train_targets.unsqueeze(1).to(device)

            with torch.no_grad():
                teacher_outputs = teacher_model(train_inputs)

            optimizer.zero_grad()      
            train_outputs = student_model(train_inputs)


            batch_loss = criterion_distillation(train_outputs, teacher_outputs, train_targets, alpha)

            batch_loss.backward()

            optimizer.step()  

            train_loss += batch_loss.item()
            train_inpsnr += batch_psnr(train_inputs, train_targets)
            train_outpsnr += batch_psnr(train_outputs, train_targets)

        train_loss /= len(train_loader)
        epoch_train_inpsnr = train_inpsnr / len(train_loader)
        epoch_train_outpsnr = train_outpsnr / len(train_loader)

        student_model.eval()
        val_loss = 0.0
        val_inpsnr = 0
        val_outpsnr = 0
        val_BM3Drltvpsnr = 0
        count = 0
        for val_inputs, val_targets in val_loader:

            val_inputs, val_targets = val_inputs.unsqueeze(1).to(device), val_targets.unsqueeze(1).to(device)

            with torch.no_grad():
                val_outputs = student_model(val_inputs)
            if batch_psnr(val_inputs, val_targets) != float('inf'):
                val_loss += criterion_inference(val_outputs, val_targets).item()
                val_inpsnr += batch_psnr(val_inputs, val_targets)
                val_outpsnr += batch_psnr(val_outputs, val_targets)
            else:
                count = count + 1

        val_loss /= (len(val_loader)-count)
        epoch_val_inpsnr = val_inpsnr / (len(val_loader)-count)
        epoch_val_outpsnr = val_outpsnr / (len(val_loader)-count)

        train_loop.set_postfix(train_loss=train_loss)


        print(f'Epoch [{epoch+1}/{epochs}], Train loss: {train_loss:.8f}, Input PSNR: {epoch_train_inpsnr:.3f}, Output PSNR: {epoch_train_outpsnr:.3f}, Alpha: {alpha:.3f}') 
        print(f'............, Validation loss: {val_loss:.8f}, Validation Input PSNR: {epoch_val_inpsnr:.3f}, Validation Output PSNR: {epoch_val_outpsnr:.3f}')

    torch.save(student_model, f'Pruned_ckpt_DistilledHARUnet_ResUNet_trainedon_CBCT_CadavarData_{(prune_amount*itr)*100}percent.pth')


Pruned 5.0% filters from ConvBlock1.block.0
Pruned 5.0% filters from ConvBlock1.block.2
Pruned 5.0% filters from ConvBlock1.conv11
Pruned 5.0% filters from pool1
Pruned 5.0% filters from ConvBlock2.block.0
Pruned 5.0% filters from ConvBlock2.block.2
Pruned 5.0% filters from ConvBlock2.conv11
Pruned 5.0% filters from pool2
Pruned 5.0% filters from ConvBlock3.block.0
Pruned 5.0% filters from ConvBlock3.block.2
Pruned 5.0% filters from ConvBlock3.conv11
Pruned 5.0% filters from pool3
Pruned 5.0% filters from ConvBlock4.block.0
Pruned 5.0% filters from ConvBlock4.block.2
Pruned 5.0% filters from ConvBlock4.conv11
Pruned 5.0% filters from pool4
Pruned 5.0% filters from ConvBlock5.block.0
Pruned 5.0% filters from ConvBlock5.block.2
Pruned 5.0% filters from ConvBlock5.conv11
Pruned 5.0% filters from ConvBlock6.block.0
Pruned 5.0% filters from ConvBlock6.block.2
Pruned 5.0% filters from ConvBlock6.conv11
Pruned 5.0% filters from ConvBlock7.block.0
Pruned 5.0% filters from ConvBlock7.block.2
Pr

Epoch 1/10: 100%|██████████| 1564/1564 [11:06<00:00,  2.35it/s]


Epoch [1/10], Train loss: 0.00007376, Input PSNR: 20.532, Output PSNR: 38.482, Alpha: 1.000
............, Validation loss: 0.00030771, Validation Input PSNR: 19.928, Validation Output PSNR: 38.026


Epoch 2/10: 100%|██████████| 1564/1564 [11:03<00:00,  2.36it/s]


Epoch [2/10], Train loss: 0.00001488, Input PSNR: 20.530, Output PSNR: 40.519, Alpha: 1.000
............, Validation loss: 0.00030305, Validation Input PSNR: 19.928, Validation Output PSNR: 38.463


...

Epoch [8/10], Train loss: 0.00000880, Input PSNR: 20.532, Output PSNR: 41.213, Alpha: 1.000
............, Validation loss: 0.00030749, Validation Input PSNR: 19.926, Validation Output PSNR: 38.669


Epoch 9/10: 100%|██████████| 1564/1564 [10:50<00:00,  2.40it/s]


Epoch [9/10], Train loss: 0.00000851, Input PSNR: 20.531, Output PSNR: 41.246, Alpha: 1.000
............, Validation loss: 0.00030707, Validation Input PSNR: 19.928, Validation Output PSNR: 38.684


Epoch 10/10: 100%|██████████| 1564/1564 [10:49<00:00,  2.41it/s]


Epoch [10/10], Train loss: 0.00000828, Input PSNR: 20.531, Output PSNR: 41.274, Alpha: 1.000
............, Validation loss: 0.00030702, Validation Input PSNR: 19.925, Validation Output PSNR: 38.708


In [8]:
for name, module in student_model.named_modules():
    if hasattr(module, "weight_orig"):
        prune.remove(module, "weight")

In [9]:
torch.save(student_model, f'Pruned_25percent_DistilledHARUnet_ResUNet_trainedon_CBCT_CadavarData_0.pth')

savedir = r"D:\CBCT_Denoising\Pickled CBCT Data"
data_dir_ = r"O:\HE_IOOS-Khuram\CBCT data cadavers\Pickled CBCT Data"
pickle_file_test = "Test_CBCT_patches.pkl"
test_inputs = load_data(os.path.join(data_dir_,pickle_file_test))  # Shape: (N, 1, H, W)
sigma = 0.03
test_targets = apply_bm3d(test_inputs, sigma=0.03)

savedir = r"D:\CBCT_Denoising\Pickled CBCT Data"
data_dir_ = r"O:\HE_IOOS-Khuram\CBCT data cadavers\Pickled CBCT Data"
pickle_file_test = "Test_CBCT_patches.pkl"
test_inputs = load_data(os.path.join(data_dir_,pickle_file_test))  # Shape: (N, 1, H, W)
sigma = 0.03
test_targets = apply_bm3d(test_inputs, sigma=0.03)

test_savefilename = os.path.join(savedir,f"Test_CBCT_inputs.pkl")
with open(test_savefilename, "wb") as f:
    pickle.dump(test_inputs, f)

test_savefilename = os.path.join(savedir,f"Test_CBCT_targets_bm3d_{sigma}.pkl")
with open(test_savefilename, "wb") as f:
    pickle.dump(test_inputs, f)



In [14]:
pickled_test_inputs = r"D:\CBCT_Denoising\Pickled CBCT Data\MoreNoisy\Test_noisyCBCT_patches.pkl"
pickled_test_targets = r"D:\CBCT_Denoising\Pickled CBCT Data\MoreNoisy\Test_origCBCT_patches.pkl"
test_inputs = load_data(pickled_test_inputs)  # Shape: (N, 1, H, W)
test_targets = load_data(pickled_test_targets)  # Shape: (N, 1, H, W)

test_dataset = CBCTDataset(test_inputs, test_targets)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True, num_workers=0,pin_memory=True)

In [19]:
model_dir = r"C:\Users\au711969\OneDrive - Aarhus universitet\Dentistry_Stuff\My Research projects\CBCT Denoising Project\BM3D_on_CBCT\Codes"

DistHARU2ResUnet_modelname = r"Pruned_25percent_DistilledHARUnet_ResUNet_trainedon_CBCT_CadavarData_0.pth"

In [ ]:

student_model = torch.load(os.path.join(model_dir,DistHARU2ResUnet_modelname)).to(device)
student_model1 = student_model.half().cuda()
student_model1.eval()
test_loss = 0.0
test_inpsnr = 0
test_outpsnr = 0
test_BM3Drltvpsnr = 0
for test_inputs, test_targets in test_loader:

    test_inputs, test_targets = test_inputs.unsqueeze(1).to(device), test_targets.unsqueeze(1).to(device)
    optimizer.zero_grad()
            
    with torch.no_grad():
        test_outputs = student_model1(test_inputs.half().cuda())
    if batch_psnr(test_inputs, test_targets) != float('inf'):
        test_loss += criterion_inference(test_outputs, test_targets).item()
        test_inpsnr += batch_psnr(test_inputs, test_targets)
        print(f'Batch_psnr = {batch_psnr(test_inputs, test_targets)}, total test_inpsnr = {test_inpsnr}')
        test_outpsnr += batch_psnr(test_outputs, test_targets)

test_loss /= (len(test_loader)-1)
epoch_test_inpsnr = test_inpsnr /(len(test_loader)-1)
epoch_test_outpsnr = test_outpsnr / (len(test_loader)-1)
epoch_test_BM3Drltvpsnr = test_BM3Drltvpsnr / (len(test_loader)-1)
        

print(f'............, Test loss: {test_loss:.8f}, Test Target PSNR: {epoch_test_inpsnr:.3f}, Test outPSNR: {epoch_test_outpsnr:.3f}')
#print(f'Validation loss: {val_loss:.8f}, Validation inPSNR: {epoch_val_inpsnr:.3f}, Validation outPSNR: {epoch_val_outpsnr:.3f}')

C:\Users\au711969\AppData\Local\Temp\ipykernel_24492\2114723937.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  student_model1 = torch.load(os.path.join(model_dir,DistHA

Batch_psnr = 19.9079714621899, total test_inpsnr = 19.9079714621899
Batch_psnr = 19.703345259729687, total test_inpsnr = 39.611316721919586
Batch_psnr = 20.705749402255385, total test_inpsnr = 60.31706612417497
Batch_psnr = 18.820504925204915, total test_inpsnr = 79.13757104937989
Batch_psnr = 19.722468009511978, total test_inpsnr = 98.86003905889186
Batch_psnr = 20.414078991655877, total test_inpsnr = 119.27411805054774
Batch_psnr = 19.55205281084455, total test_inpsnr = 138.82617086139228
Batch_psnr = 19.744657243063035, total test_inpsnr = 158.57082810445533
Batch_psnr = 19.72508524745296, total test_inpsnr = 178.2959133519083
Batch_psnr = 19.810965632910325, total test_inpsnr = 198.10687898481862
Batch_psnr = 20.149873974351756, total test_inpsnr = 218.25675295917037
Batch_psnr = 19.1320879159515, total test_inpsnr = 237.38884087512187
Batch_psnr = 20.207082662700003, total test_inpsnr = 257.5959235378219
Batch_psnr = 19.822478377194084, total test_inpsnr = 277.418401915016
Batch_p

In [13]:
# Assuming loss_history and psnr_history are your lists of metrics
data = {
    'Epoch': range(1, len(train_loss_history) + 1),  # Start epoch count from 1
    'Training Loss': train_loss_history,
    'Training BM3D PSNR': train_inpsnr_history,
    'Training outPSNR': train_outpsnr_history,
    'Validation Loss': val_loss_history,
    'Validation BM3D PSNR': val_inpsnr_history,
    'Validation outPSNR': val_outpsnr_history,
    'Testing Loss': test_loss,
    'Testing BM3D PSNR': epoch_test_inpsnr,
    'Testing outPSNR': epoch_test_outpsnr,
}

# Create a DataFrame
df = pd.DataFrame(data)

# Save to CSV
df.to_csv('TrainingnTesting_metrics_propDestilledHARUnet_epochs{epoch}_0_.csv', index=False)  # Set index=False to avoid saving row indices  